In [42]:
import pandas as pd
import random

In [44]:
class SimpleRLProductionSystem:
    def __init__(self, max_inventory, max_production, demand, epsilon=0.1):
        self.max_inventory = max_inventory
        self.max_production = max_production
        self.demand = demand
        self.epsilon = epsilon
        self.data = pd.DataFrame(columns=['State', 'Action', 'Reward'])
        self._initialize_data()

    def _initialize_data(self):
        rows = []
        for state in range(self.max_inventory + 1):
            for action in range(self.max_production + 1):
                rows.append({'State': state, 'Action': action, 'Reward': float('-inf')})
        self.data = pd.concat([self.data, pd.DataFrame(rows)], ignore_index=True)

    def choose_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.randint(0, self.max_production)  # Explore
        else:
            state_data = self.data[self.data['State'] == state]
            return state_data.loc[state_data['Reward'].idxmax()]['Action']  # Exploit

    def get_reward(self, state, action):
        inventory_after_production = min(state + action, self.max_inventory)
        if inventory_after_production >= self.demand:
            return 10 - (inventory_after_production - self.demand)  # Positive reward for meeting demand, less for overproduction
        else:
            return -10 + (inventory_after_production - self.demand)  # Negative reward for underproduction

    def update_data(self, state, action, reward):
        row_index = self.data[(self.data['State'] == state) & (self.data['Action'] == action)].index
        if self.data.at[row_index[0], 'Reward'] < reward:
            self.data.at[row_index[0], 'Reward'] = reward

    def simulate(self, episodes):
        for _ in range(episodes):
            state = random.randint(0, self.max_inventory)
            action = self.choose_action(state)
            reward = self.get_reward(state, action)
            self.update_data(state, action, reward)
            
    def evaluate_agent(self, episodes):
        total_reward = 0
        demand_satisfaction_count = 0

        for _ in range(episodes):
            state = random.randint(0, self.max_inventory)
            action = self.choose_action_test(state)
            reward = self.get_reward(state, action)
            total_reward += reward

            if self.demand == state + action:  # Exact demand satisfaction
                demand_satisfaction_count += 1

        average_reward = total_reward / episodes
        demand_satisfaction_rate = demand_satisfaction_count / episodes

        return average_reward, demand_satisfaction_rate
    
    def choose_action_test(self, state):
            state_data = self.data[self.data['State'] == state]
            return state_data.loc[state_data['Reward'].idxmax()]['Action']


In [47]:
# Example usage
system = SimpleRLProductionSystem(max_inventory=10, max_production=10, demand=5)
system.simulate(episodes=100000)
print(system.data)
system.data.to_csv('file1.csv')

    State Action  Reward
0       0      0   -15.0
1       0      1   -14.0
2       0      2   -13.0
3       0      3   -12.0
4       0      4   -11.0
..    ...    ...     ...
116    10      6     5.0
117    10      7     5.0
118    10      8     5.0
119    10      9     5.0
120    10     10     5.0

[121 rows x 3 columns]


In [49]:
# Evaluate the system
test_episodes = 100  # Number of episodes for testing
average_reward, demand_satisfaction_rate = system.evaluate_agent(episodes=test_episodes)

# Print the evaluation results
print(f"Average Reward over {test_episodes} test episodes: {average_reward}")
print(f"Demand Satisfaction Rate over {test_episodes} test episodes: {demand_satisfaction_rate}")


Average Reward over 100 test episodes: 8.84
Demand Satisfaction Rate over 100 test episodes: 0.61
